# Regional Analysis — Census Region Aggregation

Aggregates MSA-level structural gaps to the four Census regions (Northeast / Midwest / South / West), runs Kruskal-Wallis and Bonferroni-corrected pairwise significance tests, ranks top surplus/deficit markets per region, compares unweighted vs. R&D-employment-weighted rankings, and prepares regional aggregates for a choropleth map.

**Requires:** run `01_main_model.ipynb` first for the source CSV exports.

In [ ]:
"""
Census-Region Analysis (Northeast / Midwest / South / West) — v14b-Consistent
=========================================================================
Applies the same 4-region Census grouping used throughout this project
to the main model's actual output -- not a new model, pure post-hoc
regional aggregation of numbers the main model script already produces.

(Full original version history retained in git log / docs/methodology.md.)
"""

import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({"font.family": "serif", "font.size": 11})

BLUE = "#1a3a5c"
RED = "#d62728"
TEAL = "#17becf"
ORANGE = "#ff7f0e"
GREEN = "#2ca02c"

REGION_COLORS = {'Northeast': BLUE, 'Midwest': ORANGE, 'South': RED, 'West': GREEN}
REGIONS = ['Northeast', 'Midwest', 'South', 'West']

RESULTS_CSV = "AvailSFTotal_Counterfactual_Results.csv"
BYMSA_ADV_CSV = "AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv"

PREDICT_YEARS = [2020, 2021, 2022, 2023]
NON_PRESENTATION_MSAS = ['Hattiesburg, MS', 'Gulfport-Biloxi, MS']

OUTPUT_CATEGORY_CSV = "Regional_Market_Category_Distribution_v14b.csv"
OUTPUT_CATEGORY_PNG = "regional_market_category_distribution_v14b.png"
OUTPUT_YEARLY_CSV = "Regional_Yearly_Gap_Trend_v14b.csv"
OUTPUT_YEARLY_PNG = "regional_yearly_gap_trend_v14b.png"
OUTPUT_ACTUALCOUNTER_PNG = "regional_actual_vs_counterfactual_v14b.png"
OUTPUT_KW_CSV = "Regional_KruskalWallis_Test_v14b.csv"
OUTPUT_POSTHOC_CSV = "Regional_PostHoc_Pairwise_v14b.csv"
OUTPUT_BOXPLOT_PNG = "regional_structural_gap_boxplot_v14b.png"
OUTPUT_TOPMSAS_CSV = "Regional_Top5_Surplus_Deficit_v14b.csv"
OUTPUT_RDWEIGHTED_TREND_PNG = "regional_rdweighted_trend_v14b.png"
OUTPUT_RDWEIGHTED_TREND_CSV = "Regional_RDWeighted_Yearly_v14b.csv"
OUTPUT_RDWEIGHTED_FACETED_PNG = "regional_rdweighted_actual_vs_counterfactual_faceted_v14b.png"

# ══════════════════════════════════════════════════════════════════
# 0. SANITY CHECK — confirm a post-bias-fix export AND that it's a
#    true 100-MSA (v14b) export, not a 102-MSA (v14 or earlier) one.
# ══════════════════════════════════════════════════════════════════
_check = pd.read_csv(RESULTS_CSV)
_required = {'MSA_Name', 'Year', 'Structural_Gap', 'Structural_Gap_RAW',
             'Available_SF_Total', 'Counterfactual_Space_SF', 'Market_Category'}
_missing = _required - set(_check.columns)
if _missing:
    raise SystemExit(f"STOPPING -- {RESULTS_CSV} missing columns: {_missing}. "
                      f"Re-run the main model first.")
_n_msas_check = _check['MSA_Name'].nunique()
if _n_msas_check > 100:
    print(f"WARNING -- {RESULTS_CSV} contains {_n_msas_check} MSAs, not 100. This looks like "
          f"a v14 (102-MSA-trained) export, not v14b. Unlike the v13-to-v14 transition, "
          f"Sections 1-6 below ARE affected by whether the source model was trained on 100 "
          f"or 102 MSAs -- do not treat this run as v14b-equivalent if this warning fires.")
del _check

_check_adv = pd.read_csv(BYMSA_ADV_CSV)
_n_msas_check_adv = _check_adv['MSA_Name'].nunique()
if _n_msas_check_adv > 100:
    print(f"WARNING -- {BYMSA_ADV_CSV} contains {_n_msas_check_adv} MSAs, not 100. Same "
          f"v14-vs-v14b concern as above applies to Sections 7-8.")
del _check_adv

# ══════════════════════════════════════════════════════════════════
# 1. LOAD + REGION ASSIGNMENT
# ══════════════════════════════════════════════════════════════════
NORTHEAST = {'CT', 'ME', 'MA', 'NH', 'RI', 'VT', 'NJ', 'NY', 'PA'}
MIDWEST   = {'IL', 'IN', 'MI', 'OH', 'WI', 'IA', 'KS', 'MN', 'MO', 'NE', 'ND', 'SD'}
SOUTH     = {'DE', 'FL', 'GA', 'MD', 'NC', 'SC', 'VA', 'DC', 'WV',
             'AL', 'KY', 'MS', 'TN', 'AR', 'LA', 'OK', 'TX'}
WEST      = {'AZ', 'CO', 'ID', 'MT', 'NV', 'NM', 'UT', 'WY', 'AK', 'CA', 'HI', 'OR', 'WA'}

STATE_TO_REGION = {}
for s in NORTHEAST: STATE_TO_REGION[s] = 'Northeast'
for s in MIDWEST:   STATE_TO_REGION[s] = 'Midwest'
for s in SOUTH:      STATE_TO_REGION[s] = 'South'
for s in WEST:        STATE_TO_REGION[s] = 'West'

def get_primary_state(msa_name):
    if ',' not in msa_name:
        return None
    state_part = msa_name.split(',')[-1].strip().replace('/', '-')
    primary = state_part.split('-')[0].strip()
    return primary if len(primary) == 2 else None

results = pd.read_csv(RESULTS_CSV)
results = results[~results['MSA_Name'].isin(NON_PRESENTATION_MSAS)]
results['Region'] = results['MSA_Name'].map(
    lambda m: STATE_TO_REGION.get(get_primary_state(m)))

unmatched = results.loc[results['Region'].isna(), 'MSA_Name'].unique()
if len(unmatched) > 0:
    print(f"WARNING -- {len(unmatched)} MSAs unassigned to a region: {list(unmatched)}")
results = results.dropna(subset=['Region'])

print(f"Loaded {results['MSA_Name'].nunique()} MSAs assigned to regions:")
print(results.groupby('Region')['MSA_Name'].nunique().reindex(REGIONS).to_string())

predict = results[results['Year'].isin(PREDICT_YEARS)].copy()

# ══════════════════════════════════════════════════════════════════
# SECTION 1 — Market_Category distribution by region
#             [AFFECTED BY v14b -- Market_Category is a direct model
#              output; see module docstring]
# ══════════════════════════════════════════════════════════════════
cat_dist = pd.crosstab(predict['Region'], predict['Market_Category'], normalize='index') * 100
cat_dist = cat_dist.reindex(REGIONS)
cat_dist.to_csv(OUTPUT_CATEGORY_CSV)
print(f"\n{'='*70}\nMARKET CATEGORY DISTRIBUTION BY REGION (% of MSA-years, 2020-2023, 100-MSA PANEL)\n{'='*70}")
print(cat_dist.round(1).to_string())
print(f"\nSaved: {OUTPUT_CATEGORY_CSV}")

CAT_ORDER = ["Significant Structural Deficit", "Moderate Structural Deficit",
             "Structurally Balanced", "Moderate Structural Surplus", "Significant Structural Surplus"]
CAT_COLORS = {"Significant Structural Deficit": "#d62728", "Moderate Structural Deficit": "#ff7f0e",
              "Structurally Balanced": "#2ca02c", "Moderate Structural Surplus": "#1f77b4",
              "Significant Structural Surplus": "#17becf"}

fig, ax = plt.subplots(figsize=(10, 7))
bottom = np.zeros(len(REGIONS))
for cat in CAT_ORDER:
    vals = cat_dist[cat].reindex(REGIONS).fillna(0).values if cat in cat_dist.columns else np.zeros(len(REGIONS))
    ax.bar(REGIONS, vals, bottom=bottom, label=cat, color=CAT_COLORS[cat], alpha=0.88)
    bottom += vals
ax.set_ylabel("% of MSA-years, 2020-2023")
ax.set_title("Market Category Distribution by Census Region [SIZE-CORRECTED, 100-MSA PANEL]",
             fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
ax.grid(alpha=0.25, ls=':', axis='y')
plt.tight_layout()
plt.savefig(OUTPUT_CATEGORY_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_CATEGORY_PNG}")

# ══════════════════════════════════════════════════════════════════
# SECTION 2 — Regional yearly mean Structural_Gap trend, 2006-2023
#             [AFFECTED BY v14b]
# ══════════════════════════════════════════════════════════════════
yearly_region = results.groupby(['Region', 'Year'])['Structural_Gap'].mean().reset_index()
yearly_region_pivot = yearly_region.pivot(index='Year', columns='Region', values='Structural_Gap')[REGIONS]
yearly_region_pivot.to_csv(OUTPUT_YEARLY_CSV)
print(f"\nSaved: {OUTPUT_YEARLY_CSV}")

fig, ax = plt.subplots(figsize=(14, 7))
for region in REGIONS:
    ax.plot(yearly_region_pivot.index, yearly_region_pivot[region],
            'o-', color=REGION_COLORS[region], lw=2, ms=4, label=region)
ax.axhline(0, color='black', lw=0.8, ls=':')
ax.axvspan(2019.5, 2023.5, alpha=0.06, color='red', label='COVID + Recovery')
ax.set_xlabel("Year")
ax.set_ylabel("Mean Structural Gap (log units)")
ax.set_title("National Structural Gap by Census Region [SIZE-CORRECTED, 100-MSA PANEL]",
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.25, ls=':')
plt.tight_layout()
plt.savefig(OUTPUT_YEARLY_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_YEARLY_PNG}")

# ══════════════════════════════════════════════════════════════════
# SECTION 3 — Actual vs. Counterfactual, faceted by region (unweighted)
#             [AFFECTED BY v14b -- Counterfactual_Space_SF is a direct
#              model output; Available_SF_Total itself is not, but is
#              plotted alongside a series that is]
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for ax, region in zip(axes.flat, REGIONS):
    sub = results[results['Region'] == region].groupby('Year').agg(
        Actual=('Available_SF_Total', 'mean'), Counter=('Counterfactual_Space_SF', 'mean')).reset_index()
    ax.plot(sub['Year'], sub['Actual'] / 1e6, 'o-', color=REGION_COLORS[region], lw=2, ms=4, label='Actual')
    ax.plot(sub['Year'], sub['Counter'] / 1e6, 's--', color='#888888', lw=1.5, ms=3, label='Counterfactual')
    ax.fill_between(sub['Year'], sub['Actual'] / 1e6, sub['Counter'] / 1e6,
                     where=sub['Year'] >= 2020, alpha=0.13, color='#d62728')
    ax.axvspan(2019.5, 2023.5, alpha=0.05, color='red')
    ax.set_xlabel("Year")
    ax.set_ylabel("Mean Available SF Total (Million SF)")
    ax.set_title(f"{region} ({results[results['Region']==region]['MSA_Name'].nunique()} MSAs)",
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25, ls=':')
fig.suptitle("Actual vs. Counterfactual Available SF Total by Census Region [SIZE-CORRECTED, 100-MSA PANEL]",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_ACTUALCOUNTER_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_ACTUALCOUNTER_PNG}")

# ══════════════════════════════════════════════════════════════════
# SECTION 4 — Kruskal-Wallis: does Structural_Gap differ across regions?
#             [AFFECTED BY v14b]
# ══════════════════════════════════════════════════════════════════
region_groups = [predict[predict['Region'] == r]['Structural_Gap'].dropna().values for r in REGIONS]
kw_stat, kw_p = stats.kruskal(*region_groups)

print(f"\n{'='*70}")
print("KRUSKAL-WALLIS TEST -- Structural_Gap across 4 Census regions, 2020-2023 (100-MSA PANEL)")
print(f"{'='*70}")
print(f"  H-statistic = {kw_stat:.4f}, p = {kw_p:.4f}")
if kw_p < 0.05:
    print(f"  --> SIGNIFICANT: at least one region's Structural_Gap distribution differs "
          f"from the others.")
else:
    print(f"  --> NOT significant: no strong evidence the 4 regions differ overall.")

pd.DataFrame({'Metric': ['H_statistic', 'p_value', 'N_groups', 'N_total'],
              'Value': [kw_stat, kw_p, 4, sum(len(g) for g in region_groups)]}
             ).to_csv(OUTPUT_KW_CSV, index=False)
print(f"\nSaved: {OUTPUT_KW_CSV}")

# ══════════════════════════════════════════════════════════════════
# SECTION 5 — Pairwise post-hoc (Mann-Whitney U, Bonferroni-corrected)
#             [AFFECTED BY v14b]
# ══════════════════════════════════════════════════════════════════
pairs = list(combinations(REGIONS, 2))
posthoc_results = []
for r1, r2 in pairs:
    g1 = predict[predict['Region'] == r1]['Structural_Gap'].dropna()
    g2 = predict[predict['Region'] == r2]['Structural_Gap'].dropna()
    u_stat, p_raw = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    p_bonf = min(p_raw * len(pairs), 1.0)
    posthoc_results.append({
        'Region_1': r1, 'Region_2': r2, 'Median_1': g1.median(), 'Median_2': g2.median(),
        'U_statistic': u_stat, 'p_raw': p_raw, 'p_bonferroni': p_bonf,
        'Significant_at_0.05': p_bonf < 0.05,
    })

posthoc_df = pd.DataFrame(posthoc_results)
posthoc_df.to_csv(OUTPUT_POSTHOC_CSV, index=False)
print(f"\n{'='*70}")
print("PAIRWISE POST-HOC — Mann-Whitney U, Bonferroni-corrected across 6 region pairs (100-MSA PANEL)")
print(f"{'='*70}")
print(posthoc_df.round(4).to_string(index=False))
print(f"\nSaved: {OUTPUT_POSTHOC_CSV}")

# ══════════════════════════════════════════════════════════════════
# SECTION — Boxplot of Structural_Gap by region, 2020-2023
#           [AFFECTED BY v14b]
# ══════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 7))
box_data = [predict[predict['Region'] == r]['Structural_Gap'].dropna().values for r in REGIONS]
bp = ax.boxplot(box_data, labels=REGIONS, patch_artist=True, showmeans=True,
                 medianprops=dict(color='black', lw=1.5),
                 meanprops=dict(marker='D', markerfacecolor='white', markeredgecolor='black', markersize=6))
for patch, region in zip(bp['boxes'], REGIONS):
    patch.set_facecolor(REGION_COLORS[region])
    patch.set_alpha(0.6)
ax.axhline(0, color='black', lw=0.8, ls=':')
ax.set_ylabel("Structural Gap (log units)")
ax.set_title(f"Structural Gap Distribution by Region, 2020-2023 [SIZE-CORRECTED, 100-MSA PANEL]\n"
             f"Kruskal-Wallis H={kw_stat:.2f}, p={kw_p:.4f}", fontsize=12, fontweight='bold')
ax.grid(alpha=0.25, ls=':', axis='y')
plt.tight_layout()
plt.savefig(OUTPUT_BOXPLOT_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_BOXPLOT_PNG}")

# ══════════════════════════════════════════════════════════════════
# SECTION 6 — Top 5 surplus / deficit MSAs WITHIN each region
#             [AFFECTED BY v14b]
# ══════════════════════════════════════════════════════════════════
msa_avg = predict.groupby(['Region', 'MSA_Name'])['Structural_Gap'].mean().reset_index()
top_bottom_records = []
for region in REGIONS:
    sub = msa_avg[msa_avg['Region'] == region].sort_values('Structural_Gap', ascending=False)
    top5 = sub.head(5).copy(); top5['Rank_Type'] = 'Top 5 Surplus'
    bottom5 = sub.tail(5).sort_values('Structural_Gap').copy(); bottom5['Rank_Type'] = 'Top 5 Deficit'
    top_bottom_records.append(pd.concat([top5, bottom5]))

top_bottom_df = pd.concat(top_bottom_records, ignore_index=True)
top_bottom_df.to_csv(OUTPUT_TOPMSAS_CSV, index=False)
print(f"\n{'='*70}\nTOP 5 SURPLUS / DEFICIT MSAs WITHIN EACH REGION (2020-2023, 100-MSA PANEL)\n{'='*70}")
for region in REGIONS:
    print(f"\n--- {region} ---")
    sub = top_bottom_df[top_bottom_df['Region'] == region]
    print(sub[['MSA_Name', 'Structural_Gap', 'Rank_Type']].round(4).to_string(index=False))
print(f"\nSaved: {OUTPUT_TOPMSAS_CSV}")

# ══════════════════════════════════════════════════════════════════
# SECTION 7 — R&D-Weighted regional trend, 2020-2023 (line chart)
#             [AFFECTED BY v14b -- also depends on Adv_Weight_LQ,
#              which itself is unaffected by v14b (see the growth/LQ
#              script's proof), but Counterfactual_Space_SF -- one of
#              the two factors multiplied by that weight -- IS a
#              retrained model output. Net effect: affected.]
# ══════════════════════════════════════════════════════════════════
adv = pd.read_csv(BYMSA_ADV_CSV)
adv = adv.rename(columns={
    'Adv_Weighted_Available_SF_Total': 'R&D_Weighted_Available_SF_Total',
    'Adv_Weighted_Counterfactual_SF': 'R&D_Weighted_Counterfactual_SF',
})
adv = adv[~adv['MSA_Name'].isin(NON_PRESENTATION_MSAS)]
adv['Region'] = adv['MSA_Name'].map(lambda m: STATE_TO_REGION.get(get_primary_state(m)))
adv = adv.dropna(subset=['Region'])

rd_yearly = adv.groupby(['Region', 'Year']).agg(
    RD_Weighted_Actual=('R&D_Weighted_Available_SF_Total', 'mean'),
    RD_Weighted_Counter=('R&D_Weighted_Counterfactual_SF', 'mean'),
).reset_index()
rd_yearly['RD_Weighted_Gap'] = rd_yearly['RD_Weighted_Actual'] - rd_yearly['RD_Weighted_Counter']
rd_yearly.to_csv(OUTPUT_RDWEIGHTED_TREND_CSV, index=False)
print(f"\nSaved: {OUTPUT_RDWEIGHTED_TREND_CSV}")

fig, ax = plt.subplots(figsize=(14, 7))
for region in REGIONS:
    sub = rd_yearly[rd_yearly['Region'] == region]
    ax.plot(sub['Year'], sub['RD_Weighted_Gap'] / 1e6, 'o-',
            color=REGION_COLORS[region], lw=2, ms=4, label=region)
ax.axhline(0, color='black', lw=0.8, ls=':')
ax.axvspan(2019.5, 2023.5, alpha=0.06, color='red')
ax.set_xlabel("Year")
ax.set_ylabel("R&D-Weighted Structural Gap (Million SF)")
ax.set_title("R&D-Weighted Structural Gap by Region [SIZE-CORRECTED, 100-MSA PANEL]",
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.25, ls=':')
plt.tight_layout()
plt.savefig(OUTPUT_RDWEIGHTED_TREND_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_RDWEIGHTED_TREND_PNG}")

# ══════════════════════════════════════════════════════════════════
# SECTION 8 (NEW — merged from the standalone faceted script) —
#            R&D-Weighted Actual vs. Counterfactual, faceted by region
#            [AFFECTED BY v14b, same reasoning as Section 7]
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for ax, region in zip(axes.flat, REGIONS):
    sub = adv[adv['Region'] == region].groupby('Year').agg(
        Actual=('R&D_Weighted_Available_SF_Total', 'mean'),
        Counter=('R&D_Weighted_Counterfactual_SF', 'mean'),
    ).reset_index()
    ax.plot(sub['Year'], sub['Actual'] / 1e6, 'o-', color=REGION_COLORS[region], lw=2, ms=4, label='Actual')
    ax.plot(sub['Year'], sub['Counter'] / 1e6, 's--', color='#888888', lw=1.5, ms=3, label='Counterfactual')
    ax.fill_between(sub['Year'], sub['Actual'] / 1e6, sub['Counter'] / 1e6,
                     where=sub['Year'] >= 2020, alpha=0.13, color='#d62728')
    ax.axvspan(2019.5, 2023.5, alpha=0.05, color='red')
    ax.set_xlabel("Year")
    ax.set_ylabel("Mean R&D-Weighted Available SF Total (Million SF)")
    ax.set_title(f"{region} ({adv[adv['Region']==region]['MSA_Name'].nunique()} MSAs)",
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25, ls=':')
fig.suptitle("R&D-Weighted Actual vs. Counterfactual Available SF Total by Census Region\n"
             "[SIZE-CORRECTED, 100-MSA PANEL]",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_RDWEIGHTED_FACETED_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_RDWEIGHTED_FACETED_PNG}")

print(f"\n{'='*60}\nDONE -- Census-region analysis complete (v14b-consistent, 100-MSA panel; "
      f"Sections 1-8 all reflect the 100-MSA-trained model)\n{'='*60}")

In [ ]:
"""
Top 5 Structural Surplus / Deficit MSAs by Census Region — Unweighted AND
R&D-Weighted, side by side — v14b (100-MSA, true-national-LQ, size-corrected)
=========================================================================
Consolidates two previously separate pieces (Section 6 of the combined
regional-analysis script, unweighted; and the Section 6b add-on,
R&D-weighted) into one script, so both tables are guaranteed to come
from the SAME model run rather than potentially different ones.

BOTH TABLES BELOW ARE BUILT ON THE SAME UNDERLYING MODEL RUN:
  - SIZE-CORRECTED: both use the size-bias-corrected series (the
    unweighted table uses Structural_Gap directly; the R&D-weighted
    table is built from Available_SF_Total and Counterfactual_Space_SF,
    which already reflect the size-bias correction as of the main
    model's Section 8A -- see the main model's

(Full original version history retained in git log / docs/methodology.md.)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({"font.family": "serif", "font.size": 11})

BLUE = "#1a3a5c"
RED = "#d62728"
TEAL = "#17becf"
ORANGE = "#ff7f0e"
GREEN = "#2ca02c"

REGION_COLORS = {'Northeast': BLUE, 'Midwest': ORANGE, 'South': RED, 'West': GREEN}
REGIONS = ['Northeast', 'Midwest', 'South', 'West']

RESULTS_CSV = "AvailSFTotal_Counterfactual_Results.csv"
BYMSA_ADV_CSV = "AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv"

PREDICT_YEARS = [2020, 2021, 2022, 2023]
NON_PRESENTATION_MSAS = ['Hattiesburg, MS', 'Gulfport-Biloxi, MS']

OUTPUT_UNWEIGHTED_CSV = "Regional_Top5_Unweighted_v14b.csv"
OUTPUT_UNWEIGHTED_PNG = "regional_top5_unweighted_v14b.png"
OUTPUT_RDWEIGHTED_CSV = "Regional_Top5_RDWeighted_v14b.csv"
OUTPUT_RDWEIGHTED_PNG = "regional_top5_rdweighted_v14b.png"

# ══════════════════════════════════════════════════════════════════
# 0. SANITY CHECKS — required columns AND true 100-MSA (v14b) check
#    on BOTH source files
# ══════════════════════════════════════════════════════════════════
_check_results = pd.read_csv(RESULTS_CSV)
_required_results = {'MSA_Name', 'Year', 'Structural_Gap', 'Market_Category'}
_missing_results = _required_results - set(_check_results.columns)
if _missing_results:
    raise SystemExit(f"STOPPING -- {RESULTS_CSV} missing columns: {_missing_results}. "
                      f"Re-run the main model first.")
_n_msas_results = _check_results['MSA_Name'].nunique()
if _n_msas_results > 100:
    print(f"WARNING -- {RESULTS_CSV} contains {_n_msas_results} MSAs, not 100. This looks "
          f"like a v14 (102-MSA-trained) export, not v14b. The unweighted table below will "
          f"NOT reflect the 100-MSA-trained model if this warning fires.")
del _check_results

_check_adv = pd.read_csv(BYMSA_ADV_CSV)
_required_adv = {'MSA_Name', 'Year', 'Available_SF_Total', 'Adv_Weight_LQ',
                  'Adv_Weighted_Available_SF_Total', 'Adv_Weighted_Counterfactual_SF'}
_missing_adv = _required_adv - set(_check_adv.columns)
if _missing_adv:
    raise SystemExit(f"STOPPING -- {BYMSA_ADV_CSV} missing columns: {_missing_adv}. "
                      f"Re-run the main model first.")
_n_msas_adv = _check_adv['MSA_Name'].nunique()
if _n_msas_adv > 100:
    print(f"WARNING -- {BYMSA_ADV_CSV} contains {_n_msas_adv} MSAs, not 100. Same v14-vs-v14b "
          f"concern as above -- the R&D-weighted table below will NOT reflect the 100-MSA-"
          f"trained model if this warning fires.")
del _check_adv

if _n_msas_results != _n_msas_adv:
    print(f"WARNING -- {RESULTS_CSV} has {_n_msas_results} MSAs but {BYMSA_ADV_CSV} has "
          f"{_n_msas_adv}. These two files may not be from the SAME model run -- the two "
          f"tables below would then not be directly comparable. Confirm both files came "
          f"from the same v14b execution before trusting this script's output together.")

# ══════════════════════════════════════════════════════════════════
# 1. REGION ASSIGNMENT (shared method)
# ══════════════════════════════════════════════════════════════════
NORTHEAST = {'CT', 'ME', 'MA', 'NH', 'RI', 'VT', 'NJ', 'NY', 'PA'}
MIDWEST   = {'IL', 'IN', 'MI', 'OH', 'WI', 'IA', 'KS', 'MN', 'MO', 'NE', 'ND', 'SD'}
SOUTH     = {'DE', 'FL', 'GA', 'MD', 'NC', 'SC', 'VA', 'DC', 'WV',
             'AL', 'KY', 'MS', 'TN', 'AR', 'LA', 'OK', 'TX'}
WEST      = {'AZ', 'CO', 'ID', 'MT', 'NV', 'NM', 'UT', 'WY', 'AK', 'CA', 'HI', 'OR', 'WA'}

STATE_TO_REGION = {}
for s in NORTHEAST: STATE_TO_REGION[s] = 'Northeast'
for s in MIDWEST:   STATE_TO_REGION[s] = 'Midwest'
for s in SOUTH:      STATE_TO_REGION[s] = 'South'
for s in WEST:        STATE_TO_REGION[s] = 'West'

def get_primary_state(msa_name):
    if ',' not in msa_name:
        return None
    state_part = msa_name.split(',')[-1].strip().replace('/', '-')
    primary = state_part.split('-')[0].strip()
    return primary if len(primary) == 2 else None

def faceted_top5_chart(top_bottom_df, value_col, xlabel, title, output_png):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    for ax, region in zip(axes.flat, REGIONS):
        sub = top_bottom_df[top_bottom_df['Region'] == region].sort_values(value_col)
        colors = [RED if v < 0 else TEAL for v in sub[value_col]]
        ax.barh(sub['MSA_Name'], sub[value_col], color=colors, alpha=0.88,
                edgecolor='white', linewidth=0.5)
        ax.axvline(0, color='black', lw=1, ls='--')
        ax.set_xlabel(xlabel)
        ax.set_title(region, fontsize=13, fontweight='bold', color=REGION_COLORS[region])
        ax.tick_params(axis='y', labelsize=9)
        ax.grid(alpha=0.25, ls=':', axis='x')
    fig.suptitle(title, fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_png, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {output_png}")

# ══════════════════════════════════════════════════════════════════
# 2. UNWEIGHTED — Top 5 Surplus/Deficit per region, Structural_Gap
#    (log units, size-corrected)
# ══════════════════════════════════════════════════════════════════
results = pd.read_csv(RESULTS_CSV)
results = results[~results['MSA_Name'].isin(NON_PRESENTATION_MSAS)]
results['Region'] = results['MSA_Name'].map(lambda m: STATE_TO_REGION.get(get_primary_state(m)))
results = results.dropna(subset=['Region'])
predict = results[results['Year'].isin(PREDICT_YEARS)].copy()

msa_avg = predict.groupby(['Region', 'MSA_Name'])['Structural_Gap'].mean().reset_index()
top_bottom_records = []
for region in REGIONS:
    sub = msa_avg[msa_avg['Region'] == region].sort_values('Structural_Gap', ascending=False)
    top5 = sub.head(5).copy(); top5['Rank_Type'] = 'Top 5 Surplus'
    bottom5 = sub.tail(5).sort_values('Structural_Gap').copy(); bottom5['Rank_Type'] = 'Top 5 Deficit'
    top_bottom_records.append(pd.concat([top5, bottom5]))
top_bottom_df = pd.concat(top_bottom_records, ignore_index=True)
top_bottom_df.to_csv(OUTPUT_UNWEIGHTED_CSV, index=False)

print(f"\n{'='*70}")
print("TOP 5 UNWEIGHTED SURPLUS / DEFICIT MSAs WITHIN EACH REGION (2020-2023)")
print("[SIZE-CORRECTED, TRUE-NATIONAL-LQ MODEL, 100-MSA PANEL]")
print(f"{'='*70}")
for region in REGIONS:
    print(f"\n--- {region} ---")
    sub = top_bottom_df[top_bottom_df['Region'] == region]
    print(sub[['MSA_Name', 'Structural_Gap', 'Rank_Type']].round(4).to_string(index=False))
print(f"\nSaved: {OUTPUT_UNWEIGHTED_CSV}")

faceted_top5_chart(
    top_bottom_df, 'Structural_Gap', "Structural Gap (log units)",
    "Top 5 Structural Surplus / Deficit MSAs by Census Region — Unweighted, 2020-2023\n"
    "[SIZE-CORRECTED, TRUE-NATIONAL-LQ, 100-MSA PANEL]",
    OUTPUT_UNWEIGHTED_PNG,
)

# ══════════════════════════════════════════════════════════════════
# 3. R&D-WEIGHTED — Top 5 Surplus/Deficit per region, R&D_Weighted_
#    Gap_SF (SF level, size-corrected, true-national-LQ weighted)
# ══════════════════════════════════════════════════════════════════
adv = pd.read_csv(BYMSA_ADV_CSV)
adv = adv.rename(columns={
    'Adv_Weighted_Available_SF_Total': 'R&D_Weighted_Available_SF_Total',
    'Adv_Weighted_Counterfactual_SF': 'R&D_Weighted_Counterfactual_SF',
})
adv = adv[~adv['MSA_Name'].isin(NON_PRESENTATION_MSAS)]
adv['Region'] = adv['MSA_Name'].map(lambda m: STATE_TO_REGION.get(get_primary_state(m)))
adv = adv.dropna(subset=['Region'])
adv['R&D_Weighted_Gap_SF'] = adv['R&D_Weighted_Available_SF_Total'] - adv['R&D_Weighted_Counterfactual_SF']
adv_predict = adv[adv['Year'].isin(PREDICT_YEARS)]

msa_avg_rd = adv_predict.groupby(['Region', 'MSA_Name'])['R&D_Weighted_Gap_SF'].mean().reset_index()
top_bottom_records_rd = []
for region in REGIONS:
    sub = msa_avg_rd[msa_avg_rd['Region'] == region].sort_values('R&D_Weighted_Gap_SF', ascending=False)
    top5 = sub.head(5).copy(); top5['Rank_Type'] = 'Top 5 Surplus'
    bottom5 = sub.tail(5).sort_values('R&D_Weighted_Gap_SF').copy(); bottom5['Rank_Type'] = 'Top 5 Deficit'
    top_bottom_records_rd.append(pd.concat([top5, bottom5]))
top_bottom_rd_df = pd.concat(top_bottom_records_rd, ignore_index=True)
top_bottom_rd_df.to_csv(OUTPUT_RDWEIGHTED_CSV, index=False)

print(f"\n{'='*70}")
print("TOP 5 R&D-WEIGHTED SURPLUS / DEFICIT MSAs WITHIN EACH REGION (2020-2023)")
print("[SIZE-CORRECTED, TRUE-NATIONAL-LQ MODEL, 100-MSA PANEL]")
print(f"{'='*70}")
for region in REGIONS:
    print(f"\n--- {region} ---")
    sub = top_bottom_rd_df[top_bottom_rd_df['Region'] == region]
    print(sub[['MSA_Name', 'R&D_Weighted_Gap_SF', 'Rank_Type']].round(0).to_string(index=False))
print(f"\nSaved: {OUTPUT_RDWEIGHTED_CSV}")

top_bottom_rd_df['R&D_Weighted_Gap_SF_M'] = top_bottom_rd_df['R&D_Weighted_Gap_SF'] / 1e6
faceted_top5_chart(
    top_bottom_rd_df, 'R&D_Weighted_Gap_SF_M', "R&D-Weighted Structural Gap (Million SF)",
    "Top 5 R&D-Weighted Structural Surplus / Deficit MSAs by Census Region, 2020-2023\n"
    "[SIZE-CORRECTED, TRUE-NATIONAL-LQ, 100-MSA PANEL]",
    OUTPUT_RDWEIGHTED_PNG,
)

# ══════════════════════════════════════════════════════════════════
# 4. NOTE ON COMPARABILITY
# ══════════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print("NOTE: the two tables above rank DIFFERENT UNITS -- Structural_Gap is")
print("log-space; R&D_Weighted_Gap_SF is level (SF) space. Only the RANKING")
print("pattern within each region is comparable between them, not the raw")
print("magnitudes. See module docstring for detail.")
print(f"{'='*70}")

print(f"\n{'='*60}\nDONE -- Unweighted and R&D-Weighted Top-5-per-region tables complete\n{'='*60}")

In [ ]:
"""
Unweighted vs. R&D-Weighted Rank Comparison — 100-MSA Panel (v14b)
=========================================================================
Standalone script restoring a table that was present in every model
version through v14 (Section 8B's "Rank shift summary") but was
dropped from v14b's Section 8B during that script's rewrite for the
100-MSA-panel change. This does NOT retrain or recompute anything --
it just reads the two ranking CSVs v14b already exports and builds the
comparison table from them directly.

WHAT THIS SHOWS: for each MSA, how its rank changes between the
UNWEIGHTED structural-gap ranking (raw square footage, every MSA
counted equally) and the R&D-WEIGHTED ranking (each MSA's gap scaled
by its advanced-industry employment concentration, so R&D-intensive
metros count more). A large positive Rank_Shift means an MSA that
looked unremarkable in raw square-footage terms is actually a major
story once you account for how central R&D activity is to its
economy -- and vice versa for a large negative shift.

Inputs (from a v14b -- 100-MSA-trained -- run):
  AvailSFTotal_COVID_Avg_Gap.csv           (unweighted ranking source)
  AvailSFTotal_COVID_AdvWeighted_Gap.csv   (R&D-weighted ranking source)

(Full original version history retained in git log / docs/methodology.md.)
"""

import pandas as pd

UNWEIGHTED_CSV = "AvailSFTotal_COVID_Avg_Gap.csv"
WEIGHTED_CSV = "AvailSFTotal_COVID_AdvWeighted_Gap.csv"
OUTPUT_CSV = "RankComparison_Unweighted_vs_RDWeighted.csv"

# ══════════════════════════════════════════════════════════════════
# 0. LOAD + SAFETY CHECK
# ══════════════════════════════════════════════════════════════════
unweighted = pd.read_csv(UNWEIGHTED_CSV)
weighted = pd.read_csv(WEIGHTED_CSV)

_required_u = {'MSA_Name', 'Avg_Gap_2020_2023'}
_required_w = {'MSA_Name', 'Avg_Adv_Weighted_Gap_2020_2023'}
_missing_u = _required_u - set(unweighted.columns)
_missing_w = _required_w - set(weighted.columns)
if _missing_u or _missing_w:
    raise SystemExit(
        f"STOPPING -- missing columns. "
        f"{UNWEIGHTED_CSV}: {_missing_u or 'none'}. "
        f"{WEIGHTED_CSV}: {_missing_w or 'none'}."
    )

n_u, n_w = unweighted['MSA_Name'].nunique(), weighted['MSA_Name'].nunique()
if n_u > 100 or n_w > 100:
    print(f"WARNING -- {n_u} MSAs in unweighted source, {n_w} in weighted source. "
          f"Expected 100 for a v14b run -- this looks like it may be v14 (102-MSA-trained) "
          f"output instead. The comparison below will still run, but check your source "
          f"files before trusting it as the 100-MSA-trained result.")
else:
    print(f"MSA counts confirmed: {n_u} (unweighted), {n_w} (weighted) -- consistent with a "
          f"v14b 100-MSA-trained run.")

# ══════════════════════════════════════════════════════════════════
# 1. MERGE + RENAME to the project's R&D-Weighted convention
# ══════════════════════════════════════════════════════════════════
weighted = weighted.rename(columns={
    'Avg_Adv_Weighted_Gap_2020_2023': 'Avg_RD_Weighted_Gap_2020_2023',
})

comparison = unweighted.merge(weighted, on='MSA_Name', how='inner')
_dropped = set(unweighted['MSA_Name']) ^ set(weighted['MSA_Name'])
if _dropped:
    print(f"WARNING -- {len(_dropped)} MSA(s) present in only one of the two source files "
          f"and dropped from the comparison: {sorted(_dropped)}")

# ══════════════════════════════════════════════════════════════════
# 2. RANK + SHIFT
# ══════════════════════════════════════════════════════════════════
comparison['Rank_Unweighted'] = comparison['Avg_Gap_2020_2023'].rank(ascending=False)
comparison['Rank_RD_Weighted'] = comparison['Avg_RD_Weighted_Gap_2020_2023'].rank(ascending=False)
comparison['Rank_Shift'] = comparison['Rank_Unweighted'] - comparison['Rank_RD_Weighted']
# Positive Rank_Shift = moved UP (toward surplus) in the R&D-weighted
# ranking relative to the unweighted ranking; negative = moved DOWN.
comparison = comparison.sort_values('Rank_RD_Weighted').reset_index(drop=True)

comparison.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved: {OUTPUT_CSV}  ({len(comparison)} MSAs)")

# ══════════════════════════════════════════════════════════════════
# 3. PRINT — biggest movers each direction, plus the weighted top 20
# ══════════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print("RANK COMPARISON — Unweighted vs. R&D-Weighted, 100-MSA Panel (v14b)")
print(f"{'='*70}")
print(f"\nTop 20 by R&D-Weighted rank (positive Rank_Shift = moved up vs. unweighted):")
print(comparison[['MSA_Name', 'Rank_Unweighted', 'Rank_RD_Weighted', 'Rank_Shift']]
      .head(20).to_string(index=False))

print(f"\n{'='*60}")
print("BIGGEST UPWARD MOVERS (largest positive Rank_Shift)")
print("-- underrated in raw square footage, major once R&D-weighted")
print(f"{'='*60}")
print(comparison.sort_values('Rank_Shift', ascending=False)
      [['MSA_Name', 'Rank_Unweighted', 'Rank_RD_Weighted', 'Rank_Shift']]
      .head(10).to_string(index=False))

print(f"\n{'='*60}")
print("BIGGEST DOWNWARD MOVERS (largest negative Rank_Shift)")
print("-- looked significant in raw square footage, less so once R&D-weighted")
print(f"{'='*60}")
print(comparison.sort_values('Rank_Shift', ascending=True)
      [['MSA_Name', 'Rank_Unweighted', 'Rank_RD_Weighted', 'Rank_Shift']]
      .head(10).to_string(index=False))

print(f"\n{'='*60}")
print("BOTTOM 12 BY R&D-WEIGHTED RANK (largest R&D-weighted deficits)")
print("-- same Bottom-12 convention used for Top-15-surplus/Bottom-12-deficit")
print("   tables elsewhere in this project")
print(f"{'='*60}")
print(comparison.sort_values('Rank_RD_Weighted', ascending=False)
      [['MSA_Name', 'Avg_Gap_2020_2023', 'Avg_RD_Weighted_Gap_2020_2023',
        'Rank_Unweighted', 'Rank_RD_Weighted', 'Rank_Shift']]
      .head(12).to_string(index=False))

print(f"\nMean |Rank_Shift| across all {len(comparison)} MSAs: "
      f"{comparison['Rank_Shift'].abs().mean():.1f} positions")

In [ ]:
"""
Regional Choropleth Input — R&D-Weighted, Size-Corrected Structural Gap (v14b)
================================================================================
Standalone, post-hoc script. Does NOT retrain the model or touch the raw
panel — it reads the v14b model's already-exported by-MSA output and
collapses it to the 4 Census regions used throughout this project, in the
exact shape needed to drive a choropleth (one row per region, one column
per metric).

WHAT THIS PRODUCES: the region-level mean of Adv_Weighted_Gap_SF (this
project's "R&D-weighted structural gap" — Structural_Gap_SF, the
size-corrected model residual in raw SF units, multiplied by each MSA's
2015-2018 advanced-industry employment LQ weight, Adv_Weight_LQ). This is
NOT Structural_Gap in log units — it's the SF-denominated, R&D-weighted
version used in the project's Section 7/8 regional trend charts.

REQUIRES (same file the main v14b script and the by-MSA reshape script use):
  AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv

(Full original version history retained in git log / docs/methodology.md.)
"""

import pandas as pd
import numpy as np
import os

BYMSA_ADV_CSV = "AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv"
OUTPUT_CSV = "Regional_RDWeighted_Gap_For_Choropleth.csv"

PREDICT_YEARS = [2020, 2021, 2022, 2023]
NON_PRESENTATION_MSAS = ['Hattiesburg, MS', 'Gulfport-Biloxi, MS']  # no-op under v14b; kept for consistency

if not os.path.exists(BYMSA_ADV_CSV):
    raise SystemExit(
        f"STOPPING -- '{BYMSA_ADV_CSV}' not found in the working directory. "
        f"Run the main v14b model script first (it exports this file in Section 8B/8C)."
    )

# ══════════════════════════════════════════════════════════════════
# REGION ASSIGNMENT — identical mapping used throughout this project
# ══════════════════════════════════════════════════════════════════
NORTHEAST = {'CT', 'ME', 'MA', 'NH', 'RI', 'VT', 'NJ', 'NY', 'PA'}
MIDWEST   = {'IL', 'IN', 'MI', 'OH', 'WI', 'IA', 'KS', 'MN', 'MO', 'NE', 'ND', 'SD'}
SOUTH     = {'DE', 'FL', 'GA', 'MD', 'NC', 'SC', 'VA', 'DC', 'WV',
             'AL', 'KY', 'MS', 'TN', 'AR', 'LA', 'OK', 'TX'}
WEST      = {'AZ', 'CO', 'ID', 'MT', 'NV', 'NM', 'UT', 'WY', 'AK', 'CA', 'HI', 'OR', 'WA'}

STATE_TO_REGION = {}
for s in NORTHEAST: STATE_TO_REGION[s] = 'Northeast'
for s in MIDWEST:   STATE_TO_REGION[s] = 'Midwest'
for s in SOUTH:      STATE_TO_REGION[s] = 'South'
for s in WEST:        STATE_TO_REGION[s] = 'West'
REGIONS = ['Northeast', 'Midwest', 'South', 'West']

def get_primary_state(msa_name):
    if ',' not in msa_name:
        return None
    state_part = msa_name.split(',')[-1].strip().replace('/', '-')
    primary = state_part.split('-')[0].strip()
    return primary if len(primary) == 2 else None

# ══════════════════════════════════════════════════════════════════
# LOAD + RECOVER Adv_Weighted_Gap_SF
# ══════════════════════════════════════════════════════════════════
df = pd.read_csv(BYMSA_ADV_CSV)

required_cols = {"MSA_Name", "Year", "Adv_Weighted_Available_SF_Total", "Adv_Weighted_Counterfactual_SF"}
missing = required_cols - set(df.columns)
if missing:
    raise SystemExit(f"STOPPING -- '{BYMSA_ADV_CSV}' is missing expected column(s): {missing}.")

df["Adv_Weighted_Gap_SF"] = (
    df["Adv_Weighted_Available_SF_Total"] - df["Adv_Weighted_Counterfactual_SF"])

df = df[~df['MSA_Name'].isin(NON_PRESENTATION_MSAS)]
df['Region'] = df['MSA_Name'].map(lambda m: STATE_TO_REGION.get(get_primary_state(m)))

unmatched = df.loc[df['Region'].isna(), 'MSA_Name'].unique()
if len(unmatched) > 0:
    print(f"WARNING -- {len(unmatched)} MSAs unassigned to a region: {list(unmatched)}")
df = df.dropna(subset=['Region'])

print(f"Loaded {df['MSA_Name'].nunique()} MSAs assigned to regions:")
print(df.groupby('Region')['MSA_Name'].nunique().reindex(REGIONS).to_string())

# ══════════════════════════════════════════════════════════════════
# AGGREGATE TO REGION LEVEL
# ══════════════════════════════════════════════════════════════════
n_msas = df.groupby('Region')['MSA_Name'].nunique().reindex(REGIONS).rename('N_MSAs')

covid_mean = (df[df['Year'].isin(PREDICT_YEARS)]
              .groupby('Region')['Adv_Weighted_Gap_SF'].mean()
              .reindex(REGIONS).rename('Mean_RDWeighted_Gap_2020_2023'))

precovid_years = [y for y in df['Year'].unique() if y <= 2019]
precovid_mean = (df[df['Year'].isin(precovid_years)]
                  .groupby('Region')['Adv_Weighted_Gap_SF'].mean()
                  .reindex(REGIONS).rename('Mean_RDWeighted_Gap_2006_2019'))

yearly_wide = (df.groupby(['Region', 'Year'])['Adv_Weighted_Gap_SF'].mean()
               .unstack('Year').reindex(REGIONS))
yearly_wide.columns = [f'Mean_RDWeighted_Gap_{int(y)}' for y in yearly_wide.columns]

out = pd.concat([n_msas, covid_mean, precovid_mean, yearly_wide], axis=1).reset_index()

out.to_csv(OUTPUT_CSV, index=False)

print(f"\n{'='*70}")
print("REGIONAL R&D-WEIGHTED STRUCTURAL GAP -- CHOROPLETH INPUT (100-MSA PANEL)")
print(f"{'='*70}")
print(out[['Region', 'N_MSAs', 'Mean_RDWeighted_Gap_2020_2023', 'Mean_RDWeighted_Gap_2006_2019']]
      .to_string(index=False))
print(f"\nSaved: {OUTPUT_CSV}")